**Curated source notebook — archival code, not validated as end-to-end runnable.**\n\nExecution outputs and environment metadata were removed for repository hygiene. Original computational claims require independent validation; see `docs/AUDIT_STATUS.md`.\n

In [ ]:
import os, time, requests

# ---------------------------------------------------
#                CONFIGURATION
# ---------------------------------------------------
IDS_FILE = "/content/d.txt"
OUT_DIR = "/content/kloligands"
RETRIES = 3           # عدد مرات إعادة المحاولة
SLEEP_BETWEEN = 1.2   # وقت الانتظار بين كل طلب (ثواني)
TIMEOUT = 20          # المهلة القصوى للطلب
# ---------------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)

# Load ZINC IDs
with open(IDS_FILE, "r") as f:
    zinc_ids = [x.strip() for x in f if x.strip()]

session = requests.Session()

def download_zinc_sdf(zid):
    """Download SDF for a single ZINC ID with retries."""
    url = f"https://zinc20.docking.org/substances/{zid}.sdf"
    out_path = os.path.join(OUT_DIR, f"{zid}.sdf")

    for attempt in range(1, RETRIES + 1):
        try:
            r = session.get(url, timeout=TIMEOUT)

            if r.status_code == 200 and r.text.strip():
                with open(out_path, "w") as f:
                    f.write(r.text)
                print(f"✅ {zid} downloaded")
                return out_path

            elif r.status_code == 404:
                print(f"❌ {zid} not found (404)")
                return None

            else:
                print(f"⚠️ Attempt {attempt}/{RETRIES} failed for {zid} (status {r.status_code})")

        except Exception as e:
            print(f"⚠️ Error for {zid} (attempt {attempt}/{RETRIES}): {e}")

        time.sleep(SLEEP_BETWEEN)

    print(f"❌ FAILED after retries: {zid}")
    return None


successful_files = []
failed_ids = []

# ---------------------------------------------
#            DOWNLOAD ALL IDS
# ---------------------------------------------
for zid in zinc_ids:
    res = download_zinc_sdf(zid)
    if res:
        successful_files.append(res)
    else:
        failed_ids.append(zid)

# ---------------------------------------------
#            MERGE ALL SDF FILES
# ---------------------------------------------
merged_path = os.path.join(OUT_DIR, "all_ligands.sdf")

with open(merged_path, "w") as outfile:
    for sdf in successful_files:
        with open(sdf, "r") as infile:
            outfile.write(infile.read().strip() + "\n\n")

print(f"\n📦 Merged SDF saved at: {merged_path}")

# ---------------------------------------------
#            SAVE FAILED IDs
# ---------------------------------------------
if failed_ids:
    failed_file = os.path.join(OUT_DIR, "failed_ids.txt")
    with open(failed_file, "w") as f:
        f.write("\n".join(failed_ids))
    print(f"⚠️ Failed IDs saved at: {failed_file}")

print("\n🎉 DONE! All processing finished successfully.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')